In [1]:
# =======================
# 1. Install libraries
# =======================
!pip install stanza scikit-learn pandas matplotlib numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 18.6 MB/s eta 0:00:00


In [2]:
# =======================
# 2. Imports
# =======================
import os
import re
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stanza

from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

# =======================
# 3. Matplotlib style for thesis
# =======================
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10

In [3]:
# =======================
# 4. Initialize Stanza
# =======================
print("Downloading and initializing Latvian Stanza model...")
stanza.download("lv", processors="tokenize,pos,lemma")

nlp = stanza.Pipeline(
    "lv",
    processors="tokenize,pos,lemma",
    use_gpu=True,
    logging_level="WARN"
)

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading these customized packages for language: lv (Latvian)...
| Processor | Package       |
-----------------------------
| tokenize  | lvtb          |
| pos       | lvtb_nocharlm |
| lemma     | lvtb_nocharlm |
| pretrain  | conll17       |



INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/lv/tokenize/lvtb.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/lv/pos/lvtb_nocharlm.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/lv/lemma/lvtb_nocharlm.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/lv/pretrain/conll17.pt
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


In [4]:
# =======================
# 5. Upload ZIP file
# =======================
print("\n--- UPLOAD ZIP FILE ---")
print("Please upload a ZIP archive with leaflet .txt files.")
uploaded = files.upload()

if not uploaded:
    raise SystemExit("No file uploaded.")

zip_name = next(iter(uploaded.keys()))

extract_dir = "leaflets_zip"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(io.BytesIO(uploaded[zip_name]), "r") as zf:
    zf.extractall(extract_dir)

text_files = [
    os.path.join(extract_dir, f)
    for f in os.listdir(extract_dir)
    if f.lower().endswith(".txt")
]

print(f"Successfully extracted {len(text_files)} text files.")


--- UPLOAD ZIP FILE ---
Please upload a ZIP archive with leaflet .txt files.


Saving LCLC_1934-1940_only_unique_text_leaflets.zip to LCLC_1934-1940_only_unique_text_leaflets.zip
Successfully extracted 251 text files.


In [ ]:
# =======================
# 6. Configuration
# =======================
TARGET_YEARS = list(range(1934, 1940))  # 1934–1939 only
N_TOPICS = 6
TOP_WORDS_PER_TOPIC = 15

LATVIAN_STOP_LEMMAS = set("""
un ir būt tikt kļūt par lai kas bet arī kā tas šis
pie pret vai kad tad tik ar no uz mēs jūs viņš viņa
kurš kāds viss jo gan vēl jau pat kurp ne nav
ļoti tikai nekā taču man tevi sev savs
""".split())

META_STOP = {
    "id", "file_name", "title", "author", "date", "source", "text",
    "print_run", "typography_name", "production_method"
}

STOPWORDS = LATVIAN_STOP_LEMMAS | META_STOP

ALLOWED_POS = {"NOUN", "VERB", "ADJ", "PROPN"}

topic_labels = {
    0: "Antifašisms un Spānija",
    1: "Strādnieku problēmas",
    2: "Jaunatnes problēmas",
    3: "Sieviešu tiesības",
    4: "Politieslodzīto palīdzība",
    5: "Karš un ārpolitika"
}

In [ ]:
# =======================
# 7. Helper functions
# =======================
def extract_text_block(full_text: str) -> str:
    parts = re.split(r"(?im)^\s*text\s*:\s*", full_text, maxsplit=1)
    text = parts[1] if len(parts) == 2 else full_text
    return text.strip()


def extract_year(full_text: str, fallback_filename: str = ""):
    m = re.search(r"(?im)^\s*date\s*:\s*(.+)$", full_text)
    if m:
        date_value = m.group(1).strip()
        y = re.search(r"(19\d{2}|20\d{2})", date_value)
        if y:
            return int(y.group(1))

    m2 = re.search(r"(?im)^\s*file_name\s*:\s*(.+)$", full_text)
    if m2:
        file_value = m2.group(1).strip()
        y2 = re.search(r"(19\d{2}|20\d{2})", file_value)
        if y2:
            return int(y2.group(1))

    if fallback_filename:
        y3 = re.search(r"(19\d{2}|20\d{2})", fallback_filename)
        if y3:
            return int(y3.group(1))

    return None


def extract_print_run(full_text: str, fallback_filename: str = ""):
    m = re.search(r"(?im)^\s*print_run\s*:\s*(.+)$", full_text)
    if m:
        value = m.group(1).strip().lower()
        if value == "unk":
            return np.nan
        if re.fullmatch(r"\d+", value):
            return float(value)

    base = os.path.basename(fallback_filename)
    m2 = re.search(r"-(\d+)-[\[\.…]", base)
    if m2:
        return float(m2.group(1))

    return np.nan


def normalize_lemma(lemma: str) -> str:
    lemma = lemma.lower().strip()

    if lemma.startswith("sociāldemokr"):
        return "sociāldemokrāts"
    if lemma.startswith("fašist"):
        return "fašists"
    if lemma.startswith("komunist"):
        return "komunists"
    if lemma.startswith("strādniek"):
        return "strādnieks"
    if lemma.startswith("bezdarbniek"):
        return "bezdarbnieks"
    if lemma.startswith("kapitālist"):
        return "kapitālists"
    if lemma.startswith("buržuāz"):
        return "buržuāzija"

    return lemma


def lemmatize_text(full_text: str) -> str:
    text = extract_text_block(full_text)

    text = re.sub(r"s\.\-d\.?", " sociāldemokrāti ", text, flags=re.IGNORECASE)
    text = re.sub(r"1\.\s*maijs", " pirmais_maijs ", text, flags=re.IGNORECASE)
    text = re.sub(r"7\.\s*novembris", " septītais_novembris ", text, flags=re.IGNORECASE)
    text = re.sub(r"padomju latvija", " padomju_latvija ", text, flags=re.IGNORECASE)
    text = re.sub(r"politiskā pārvalde", " politiskā_pārvalde ", text, flags=re.IGNORECASE)

    if len(text) > 100000:
        text = text[:100000]

    doc = nlp(text)

    lemmas = []
    for sent in doc.sentences:
        for word in sent.words:
            lemma = normalize_lemma(word.lemma or "")
            if (
                len(lemma) > 2
                and lemma not in STOPWORDS
                and word.upos in ALLOWED_POS
                and re.fullmatch(r"[a-zāčēģīķļņšūž_]+", lemma)
            ):
                lemmas.append(lemma)

    return " ".join(lemmas)

In [ ]:
# =======================
# 8. Process files
# =======================
data_rows = []
failed_rows = []

print(f"\nStarting preprocessing of {len(text_files)} files...")

for i, filepath in enumerate(text_files, start=1):
    try:
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            raw_content = f.read()

        year = extract_year(raw_content, fallback_filename=filepath)

        if year not in TARGET_YEARS:
            failed_rows.append({
                "file_name": os.path.basename(filepath),
                "reason": "year_outside_target_or_missing"
            })
            continue

        clean_content = lemmatize_text(raw_content)

        if not clean_content.strip():
            failed_rows.append({
                "file_name": os.path.basename(filepath),
                "reason": "empty_after_lemmatization"
            })
            continue

        print_run = extract_print_run(raw_content, fallback_filename=filepath)

        data_rows.append({
            "file_name": os.path.basename(filepath),
            "year": year,
            "print_run": print_run,
            "raw_text": raw_content,
            "clean_text": clean_content
        })

        if i % 10 == 0 or i == len(text_files):
            print(f"Processed {i}/{len(text_files)} files... accepted: {len(data_rows)}")

    except Exception as e:
        failed_rows.append({
            "file_name": os.path.basename(filepath),
            "reason": f"exception: {e}"
        })

df = pd.DataFrame(data_rows)
failed_df = pd.DataFrame(failed_rows)

print("\nPreprocessing complete.")
print(f"Accepted documents: {len(df)}")
print(f"Excluded documents: {len(failed_df)}")

print("\nDocuments per year:")
print(df["year"].value_counts().sort_index())

df.to_csv("leaflets_preprocessed_1934_1939.csv", index=False, encoding="utf-8-sig")
failed_df.to_csv("leaflets_excluded_diagnostics.csv", index=False, encoding="utf-8-sig")

In [ ]:
# =======================
# 9. TF-IDF vectorization
# =======================
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_tfidf = tfidf_vectorizer.fit_transform(df["clean_text"])
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"\nTF-IDF matrix shape: {X_tfidf.shape}")

# =======================
# 10. NMF topic modeling
# =======================
nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=42,
    init="nndsvda",
    max_iter=1000
)

W = nmf_model.fit_transform(X_tfidf)
H = nmf_model.components_

print(f"\nTop words per topic (N={N_TOPICS}):")
topic_summaries = []

for topic_idx, topic in enumerate(H):
    top_indices = topic.argsort()[::-1][:TOP_WORDS_PER_TOPIC]
    top_words = [feature_names[i] for i in top_indices]
    summary = ", ".join(top_words)
    topic_summaries.append(summary)
    print(f"Topic {topic_idx}: {summary}")

In [ ]:
# =======================
# 11. Assign topics to documents
# =======================
df["topic_id"] = W.argmax(axis=1)
df["topic_score"] = W.max(axis=1)
df["topic_keywords"] = df["topic_id"].apply(lambda x: topic_summaries[x])
df["topic_label"] = df["topic_id"].map(topic_labels)

df.to_csv("latvian_leaflets_nmf_topics_1934_1939.csv", index=False, encoding="utf-8-sig")
print("\nSaved: latvian_leaflets_nmf_topics_1934_1939.csv")

topic_table = pd.DataFrame({
    "topic_id": list(range(N_TOPICS)),
    "topic_label": [topic_labels[i] for i in range(N_TOPICS)],
    "top_keywords": topic_summaries
})

topic_table.to_csv("nmf_topic_keywords_1934_1939.csv", index=False, encoding="utf-8-sig")
print("Saved: nmf_topic_keywords_1934_1939.csv")

In [ ]:
# =======================
# 12. Plot 1: Number of documents per topic
# =======================
topic_counts = df["topic_id"].value_counts().sort_index()
topic_counts.index = [topic_labels[i] for i in topic_counts.index]

plt.figure(figsize=(12, 6))
topic_counts.plot(kind="bar", edgecolor="black")
plt.title("Skrejlapu skaits pa tēmām (1934–1939)")
plt.xlabel("Tēmas")
plt.ylabel("Skrejlapu skaits")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("01_documents_per_topic_1934_1939.png", dpi=600, bbox_inches="tight")
plt.show()

# =======================
# 13. Plot 2: Topic share by year (dominant topic)
# =======================
pivot_counts = df.groupby(["year", "topic_id"]).size().unstack(fill_value=0)
pivot_counts = pivot_counts.reindex(TARGET_YEARS, fill_value=0)
pivot_counts = pivot_counts.rename(columns=topic_labels)

pivot_pct = pivot_counts.div(pivot_counts.sum(axis=1), axis=0).fillna(0)

ax = pivot_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 7),
    colormap="tab10",
    width=0.85,
    edgecolor="none"
)

plt.title("Tēmu īpatsvars pa gadiem (1934–1939)")
plt.xlabel("Gads")
plt.ylabel("Īpatsvars")
plt.legend(title="Tēmas", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("02_topic_share_by_year_1934_1939.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# =======================
# 14. Plot 3: Soft clustering topic share by year
# =======================
topic_weights_df = pd.DataFrame(W, columns=[topic_labels[i] for i in range(N_TOPICS)])
topic_weights_df["year"] = df["year"].values

soft_clustering = topic_weights_df.groupby("year").sum()
soft_clustering = soft_clustering.reindex(TARGET_YEARS, fill_value=0)

soft_clustering_pct = soft_clustering.div(soft_clustering.sum(axis=1), axis=0).fillna(0)

ax = soft_clustering_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 7),
    colormap="tab10",
    width=0.85,
    edgecolor="none"
)

plt.title("Tēmu īpatsvars (soft clustering) (1934–1939)")
plt.ylabel("Tēmu īpatsvars")
plt.xlabel("Gads")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Tēmas")
plt.tight_layout()
plt.savefig("03_soft_clustering_share_1934_1939.png", dpi=600, bbox_inches="tight")
plt.show()

# =======================
# 15. Plot 4: Absolute topic intensity by year
# =======================
ax = soft_clustering.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 7),
    colormap="tab10",
    width=0.85,
    edgecolor="none"
)

plt.title("Ideoloģiskā dinamika (1934–1939): tēmu intensitāte")
plt.ylabel("Tēmu svaru summa")
plt.xlabel("Gads")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Tēmas")
plt.tight_layout()
plt.savefig("04_topic_intensity_absolute_1934_1939.png", dpi=600, bbox_inches="tight")
plt.show()

# =======================
# 16. Plot 5: Print-run-weighted topic intensity
# =======================
median_print_run = df["print_run"].median()
df["print_run_filled"] = df["print_run"].fillna(median_print_run)

print(f"\nMedian print run used for missing values: {median_print_run}")

topic_weights_weighted = pd.DataFrame(
    W,
    columns=[topic_labels[i] for i in range(N_TOPICS)]
)

topic_weights_weighted = topic_weights_weighted.multiply(df["print_run_filled"], axis=0)
topic_weights_weighted["year"] = df["year"].values

impact_data = topic_weights_weighted.groupby("year").sum()
impact_data = impact_data.reindex(TARGET_YEARS, fill_value=0)

ax = impact_data.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 7),
    colormap="tab10",
    width=0.85,
    edgecolor="none"
)

plt.title("Tēmu tirāžas apjoms (1934–1939)")
plt.ylabel("Aprēķinātais kopējais tirāžas svars")
plt.xlabel("Gads")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Tēmas")
plt.tight_layout()
plt.savefig("05_topic_print_run_weighted_1934_1939.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# =======================
# 17. Elbow method
# =======================
n_range = range(2, 16)
reconstruction_errors = []

print("\nCalculating reconstruction errors for NMF...")

for n in n_range:
    nmf_test = NMF(
        n_components=n,
        init="nndsvda",
        random_state=42,
        max_iter=500
    )
    nmf_test.fit(X_tfidf)
    reconstruction_errors.append(nmf_test.reconstruction_err_)
    print(f"Checked N={n}, Error={nmf_test.reconstruction_err_:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(list(n_range), reconstruction_errors, marker="o")
plt.title("Elbow method for optimal number of topics")
plt.xlabel("Number of topics (N)")
plt.ylabel("Reconstruction error")
plt.grid(True)
plt.tight_layout()
plt.savefig("06_nmf_elbow_1934_1939.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# =======================
# 18. Lemmatization quality check
# =======================
def check_lemmatization_quality(search_query: str):
    found_row = df[df["file_name"].astype(str).str.contains(search_query, case=False, regex=False)]

    if found_row.empty:
        print(f"No file found containing: {search_query}")
        return

    row = found_row.iloc[0]

    print(f"\nFILE: {row['file_name']}")
    print("=" * 70)

    print("\n[RAW TEXT SAMPLE]")
    print(row["raw_text"][:500].replace("\n", " ") + "...")

    print("\n[CLEAN TEXT SAMPLE]")
    print(row["clean_text"][:500] + "...")

    print("\n[FIRST 30 LEMMAS]")
    print(row["clean_text"].split()[:30])

check_lemmatization_quality("1934")

test_phrase = "Kārļa Ulmaņa fašistiskā valdība, kas bendē tautu, atkal palielina nodokļus!"
processed = lemmatize_text(test_phrase)

print("\nTest phrase:")
print(test_phrase)
print("\nLemmatized:")
print(processed)

In [ ]:
# =======================
# 19. Download key result files
# =======================
files.download("latvian_leaflets_nmf_topics_1934_1939.csv")
files.download("nmf_topic_keywords_1934_1939.csv")
files.download("leaflets_excluded_diagnostics.csv")